In [26]:
import geopandas as gpd
import pandas as pd
import vaex

In [27]:
ano = 2020

In [28]:
agg_atributos = {
        'Quantidade de Unidades':'sum',
        'Quantidade de Unidades Condominiais':'sum',
        'Tamanho Médio da Unidade Condominial':'mean',
        'Tamanho médio dos Terrenos':'mean',
        'Área Total dos lotes':'sum',
        'Área Total Ocupada':'sum',
        'Área Total Construída':'sum',
        'Valor Total dos Terrenos':'sum',
        'Valor Total das Construções':'sum',
        'CA médio':'mean',
        'TO médio':'mean',
        'CA médio em lotes condominiais':'mean',
        'TO médio em lotes condominiais':'mean',
        'CA médio em lotes não condominiais':'mean',
        'TO médio em lotes não condominiais':'mean',
        'Comprimento Médio da Testada':'mean',
        'Número médio de Pavimentos':'mean',
        'Fator de obsolecência médio':'mean',
        'Residencial vertical Baixo (m2)':'sum',
        'Residencial vertical Médio (m2)':'sum',
        'Residencial vertical Alto (m2)':'sum',
        'Residencial horizontal Baixo (m2)':'sum',
        'Residencial horizontal Médio (m2)':'sum',
        'Residencial horizontal Alto (m2)':'sum',
        'Comercial vertical Baixo (m2)':'sum',
        'Comercial vertical Médio (m2)':'sum',
        'Comercial vertical Alto (m2)':'sum',
        'Comercial horizontal Baixo (m2)':'sum',
        'Comercial horizontal Alto (m2)':'sum',
        'Comercial horizontal Médio (m2)':'sum',
        'Terreno (m2)':'sum',
        'Outros Usos (m2)':'sum',
        'Residencial vertical Baixo (qt)':'sum',
        'Residencial vertical Médio (qt)':'sum',
        'Residencial vertical Alto (qt)':'sum',
        'Residencial horizontal Baixo (qt)':'sum',
        'Residencial horizontal Médio (qt)':'sum',
        'Residencial horizontal Alto (qt)':'sum',
        'Comercial vertical Baixo (qt)':'sum',
        'Comercial vertical Médio (qt)':'sum',
        'Comercial vertical Alto (qt)':'sum',
        'Comercial horizontal Baixo (qt)':'sum',
        'Comercial horizontal Alto (qt)':'sum',
        'Comercial horizontal Médio (qt)':'sum',
        'Terreno (qt)':'sum',
        'Outros Usos (qt)':'sum'
}

In [29]:
gdf_distritos = gpd.read_file('data/SIRGAS_GPKG_distrito.gpkg')

In [49]:
for distrito in gdf_distritos.iterrows():
    print(distrito[1].ds_nome)
    distrito = gdf_distritos[gdf_distritos.ds_codigo == distrito[1].ds_codigo].iloc[0]
    path = f'lotes_agregados_por_ano/{ano}/SIRGAS_SHP_LOTES_{distrito.ds_codigo.rjust(2, "0")}_{distrito.ds_nome.replace(" ", "_")}_IPTU_{ano}.gpkg'
    gdf_lote = gpd.read_file(path).drop_duplicates(subset=['sqlc']).set_index('sqlc')
    df_iptu = vaex.open(f'data/por_distritos/IPTU-1995-{2024}-agrupados-por-sqlc-{distrito.ds_codigo}-{distrito.ds_nome.replace(" ", "-").lower()}.hdf5').to_pandas_df().set_index('sqlc')
    df_iptu = df_iptu[df_iptu.ano == ano]
    lotes_existentes = gdf_lote.join(df_iptu, how='inner')
    lotes_sg = df_iptu.join(gdf_lote, how='left').sq.isna()
    df_lotes_sg = df_iptu[lotes_sg].reset_index()
    df_lotes_sg.sqlc = df_lotes_sg.sqlc.str[:6] + '000000'
    df_lotes_sg_group = df_lotes_sg.groupby('sqlc').agg(agg_atributos)
    lotes_agregados = gdf_lote.join(df_lotes_sg_group, how='inner')
    lotes = pd.concat([lotes_existentes, lotes_agregados])
    lotes.to_file(f"iptu_por_lotes/{ano}/IPTU-SP-todos-atributos-por-lotes-{distrito.ds_codigo}-{distrito.ds_nome.lower().replace(' ', '-')}.gpkg", driver='GPKG')
    # break

PIRITUBA
SAO DOMINGOS
JARAGUA
BRASILANDIA
FREGUESIA DO O
CASA VERDE
CACHOEIRINHA
LIMAO
VILA GUILHERME
VILA MARIA
VILA MEDEIROS
ARTUR ALVIM
PENHA
CANGAIBA
VILA MATILDE
PONTE RASA
ERMELINO MATARAZZO
VILA CURUCA
ITAIM PAULISTA
GUAIANASES
LAJEADO
BARRA FUNDA
PERDIZES
VILA LEOPOLDINA
JAGUARA
LAPA
JAGUARE
REPUBLICA
SANTA CECILIA
SE
BELA VISTA
BOM RETIRO
CAMBUCI
CONSOLACAO
LIBERDADE
MOOCA
PARI
TATUAPE
AGUA RASA
BELEM
BRAS
CARRAO
VILA FORMOSA
ARICANDUVA
SAO MATEUS
SAO RAFAEL
IGUATEMI
VILA PRUDENTE
SAO LUCAS
MORUMBI
RIO PEQUENO
VILA SONIA
BUTANTA
RAPOSO TAVARES
PINHEIROS
ALTO DE PINHEIROS
ITAIM BIBI
JARDIM PAULISTA
CAMPO LIMPO
CAPAO REDONDO
VILA ANDRADE
JARDIM ANGELA
JARDIM SAO LUIS
SOCORRO
CIDADE DUTRA
GRAJAU
MARSILAC
PARELHEIROS
CIDADE TIRADENTES
PERUS
ANHANGUERA
SAPOPEMBA
SACOMA
CURSINO
IPIRANGA
MOEMA
SAUDE
VILA MARIANA
PEDREIRA
CIDADE ADEMAR
JACANA
TREMEMBE
MANDAQUI
SANTANA
TUCURUVI
SANTO AMARO
CAMPO GRANDE
CAMPO BELO
JABAQUARA
VILA JACUI
SAO MIGUEL
JARDIM HELENA
CIDADE LIDER
PARQUE DO CARM

In [34]:
df_iptu = df_iptu[df_iptu.ano == ano]
lotes_existentes = gdf_lote.join(df_iptu, how='inner')
lotes_sg = df_iptu.join(gdf_lote, how='left').sq.isna()
df_lotes_sg = df_iptu[lotes_sg].reset_index()
df_lotes_sg.sqlc = df_lotes_sg.sqlc.str[:6] + '000000'
df_lotes_sg_group = df_lotes_sg.groupby('sqlc').agg(agg_atributos)
lotes_agregados = gdf_lote.join(df_lotes_sg_group, how='inner')
lotes = pd.concat([lotes_existentes, lotes_agregados])

In [37]:
lotes

,sq,agregado,geometry,ano,Quantidade de Unidades,Quantidade de Unidades Condominiais,Tamanho Médio da Unidade Condominial,Tamanho médio dos Terrenos,Área Total dos lotes,Área Total Ocupada,...,Residencial horizontal Médio (qt),Residencial horizontal Alto (qt),Comercial vertical Baixo (qt),Comercial vertical Médio (qt),Comercial vertical Alto (qt),Comercial horizontal Baixo (qt),Comercial horizontal Alto (qt),Comercial horizontal Médio (qt),Terreno (qt),Outros Usos (qt)
sqlc,,,,,,,,,,,,,,,,,,,,,
052246000004,052246,False,"POLYGON ((339259.190 7391880.414, 339258.522 7...",2020.0,5,5,104.400000,305.0,305.0,222.0,...,5,0,0,0,0,0,0,0,0,0
102087000002,102087,False,"POLYGON ((339975.061 7391373.027, 339972.832 7...",2020.0,79,79,115.911392,1941.0,1941.0,1345.0,...,0,0,0,0,0,0,0,0,0,0
052062000002,052062,False,"POLYGON ((338387.249 7393011.755, 338342.440 7...",2020.0,192,192,120.265625,5242.0,5242.0,3232.0,...,0,0,0,0,0,0,0,0,0,0
100069002500,100069,False,"POLYGON ((339511.610 7391704.014, 339516.247 7...",2020.0,1,0,NaN,320.0,320.0,150.0,...,1,0,0,0,0,0,0,0,0,0
100017002100,100017,False,"POLYGON ((338846.759 7391897.295, 338849.031 7...",2020.0,1,0,NaN,125.0,125.0,80.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102086000000,102086,True,"POLYGON ((340017.654 7391568.253, 340015.544 7...",NaN,1,0,NaN,205.0,205.0,108.0,...,1,0,0,0,0,0,0,0,0,0
102088000000,102088,True,"MULTIPOLYGON (((340288.254 7391299.778, 340289...",NaN,5,0,NaN,96.0,480.0,248.0,...,5,0,0,0,0,0,0,0,0,0
102099000000,102099,True,"MULTIPOLYGON (((340717.177 7391452.895, 340694...",NaN,1,0,NaN,3230.0,3230.0,0.0,...,0,0,0,0,0,0,0,0,1,0


In [36]:
df_iptu

,ano,Quantidade de Unidades,Quantidade de Unidades Condominiais,Tamanho Médio da Unidade Condominial,Tamanho médio dos Terrenos,Área Total dos lotes,Área Total Ocupada,Área Total Construída,Valor Total dos Terrenos,Valor Total das Construções,...,Residencial horizontal Médio (qt),Residencial horizontal Alto (qt),Comercial vertical Baixo (qt),Comercial vertical Médio (qt),Comercial vertical Alto (qt),Comercial horizontal Baixo (qt),Comercial horizontal Alto (qt),Comercial horizontal Médio (qt),Terreno (qt),Outros Usos (qt)
sqlc,,,,,,,,,,,,,,,,,,,,,
029095002900,2020,1,0,NaN,146.0,146.0,119.0,119,185712.0,162792.0,...,1,0,0,0,0,0,0,0,0,0
029098000200,2020,1,0,NaN,256.0,256.0,90.0,120,341760.0,160920.0,...,0,0,0,0,0,1,0,0,0,0
031076004400,2020,1,0,NaN,104.0,104.0,50.0,100,178776.0,107400.0,...,0,0,0,0,0,0,0,0,0,0
031082000200,2020,1,0,NaN,200.0,200.0,170.0,340,317000.0,455940.0,...,0,0,0,0,0,1,0,0,0,0
031086001800,2020,1,0,NaN,170.0,170.0,80.0,205,268940.0,220170.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102053005900,2020,1,0,NaN,125.0,125.0,75.0,146,140125.0,199728.0,...,1,0,0,0,0,0,0,0,0,0
052050030500,2020,1,0,NaN,500.0,500.0,345.0,477,694000.0,414036.0,...,0,0,0,0,0,0,0,0,0,0
031094027600,2020,1,0,NaN,229.0,229.0,119.0,223,387697.0,305064.0,...,1,0,0,0,0,0,0,0,0,0
